In [10]:
# =========================
# 2. LOAD DATA
# =========================

financial_df = pd.read_csv(financial_path)

# Fix UnicodeDecodeError for SCMS dataset
scms_df = pd.read_csv(
    scms_path,
    encoding="latin1",
    low_memory=False
)

print("Financial shape:", financial_df.shape)
print("SCMS shape:", scms_df.shape)

print("\nFinancial dataset:")
display(financial_df.head())

print("\nSCMS dataset:")
display(scms_df.head())

Financial shape: (500, 4)
SCMS shape: (10324, 33)

Financial dataset:


,Date,Expense_Category,Amount,Description
0,2024-10-01,Staffing,29391.86,Surgical masks
1,2024-10-02,Supplies,47757.71,Surgical masks
2,2024-10-03,Supplies,43996.60,Ventilators
3,2024-10-04,Supplies,27908.42,Surgeons' salaries
4,2024-10-05,Equipment,39719.60,Ventilators



SCMS dataset:


,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,Unit of Measure (Per Pack),Line Item Quantity,Line Item Value,Pack Price,Unit Price,Manufacturing Site,First Line Designation,Weight (Kilograms),Freight Cost (USD),Line Item Insurance (USD)
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,...,30,19,551.0,29.00,0.97,Ranbaxy Fine Chemicals LTD,Yes,13,780.34,NaN
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,...,240,1000,6200.0,6.20,0.03,"Aurobindo Unit III, India",Yes,358,4521.5,NaN
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,...,100,500,40000.0,80.00,0.80,ABBVIE GmbH & Co.KG Wiesbaden,Yes,171,1653.78,NaN
3,15,108-VN-T01,Pre-PQ Process,SCMS-78,ASN-50,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,31920,127360.8,3.99,0.07,"Ranbaxy, Paonta Shahib, India",Yes,1855,16007.06,NaN
4,16,108-VN-T01,Pre-PQ Process,SCMS-81,ASN-55,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,38000,121600.0,3.20,0.05,"Aurobindo Unit III, India",Yes,7590,45450.08,NaN


In [11]:
# STEP 3: CHECK AND CLEAN DATA

print("Financial missing values:")
print(financial_df.isnull().sum())

print("\nSCMS missing values:")
print(scms_df.isnull().sum())

print("\nFinancial data types:")
print(financial_df.dtypes)

print("\nSCMS data types:")
print(scms_df.dtypes)

Financial missing values:
Date                0
Expense_Category    0
Amount              0
Description         0
dtype: int64

SCMS missing values:
ID                                 0
Project Code                       0
PQ #                               0
PO / SO #                          0
ASN/DN #                           0
Country                            0
Managed By                         0
Fulfill Via                        0
Vendor INCO Term                   0
Shipment Mode                    360
PQ First Sent to Client Date       0
PO Sent to Vendor Date             0
Scheduled Delivery Date            0
Delivered to Client Date           0
Delivery Recorded Date             0
Product Group                      0
Sub Classification                 0
Vendor                             0
Item Description                   0
Molecule/Test Type                 0
Brand                              0
Dosage                          1736
Dosage Form                        0


In [13]:
# STEP 3: DATA PREPROCESSING

import pandas as pd
import numpy as np

financial_df["Date"] = pd.to_datetime(
    financial_df["Date"], errors="coerce"
)

# Fill missing financial text values
for col in financial_df.select_dtypes(
    include=["object", "str"]
).columns:
    financial_df[col] = financial_df[col].fillna("Unknown")

financial_df["Amount"] = pd.to_numeric(
    financial_df["Amount"], errors="coerce"
)
financial_df["Amount"] = financial_df["Amount"].fillna(
    financial_df["Amount"].median()
)

# Fill missing SCMS values
for col in scms_df.columns:
    if scms_df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(scms_df[col]):
            scms_df[col] = scms_df[col].fillna(
                scms_df[col].median()
            )
        else:
            scms_df[col] = scms_df[col].fillna("Unknown")

print("Financial missing values:", financial_df.isnull().sum().sum())
print("SCMS missing values:", scms_df.isnull().sum().sum())
print("Financial shape:", financial_df.shape)
print("SCMS shape:", scms_df.shape)

Financial missing values: 0
SCMS missing values: 0
Financial shape: (500, 4)
SCMS shape: (10324, 33)


In [15]:
# FIX: Parse delivery dates safely

date_cols = [
    "Scheduled Delivery Date",
    "Delivered to Client Date"
]

for col in date_cols:
    scms_df[col] = pd.to_datetime(
        scms_df[col],
        format="mixed",
        errors="coerce"
    )

# Calculate delivery delay in days
scms_df["Delivery_Delay_Days"] = (
    scms_df["Delivered to Client Date"]
    - scms_df["Scheduled Delivery Date"]
).dt.days

print("Date conversion completed!")
print("\nDelivery delay results:")
print(
    scms_df[
        [
            "Scheduled Delivery Date",
            "Delivered to Client Date",
            "Delivery_Delay_Days"
        ]
    ].head()
)

print("\nMissing delivery delays:",
      scms_df["Delivery_Delay_Days"].isna().sum())

Date conversion completed!

Delivery delay results:
  Scheduled Delivery Date Delivered to Client Date  Delivery_Delay_Days
0              2006-06-02               2006-06-02                    0
1              2006-11-14               2006-11-14                    0
2              2006-08-27               2006-08-27                    0
3              2006-09-01               2006-09-01                    0
4              2006-08-11               2006-08-11                    0

Missing delivery delays: 0


In [16]:
# STEP 5: TRAIN STOCKSENTRY DELAY PREDICTION MODEL

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create a separate modeling dataset
model_df = scms_df.copy()

# Create target: 1 = delayed, 0 = on time or early
model_df = model_df.dropna(
    subset=["Scheduled Delivery Date",
            "Delivered to Client Date"]
)

model_df["Delay_Days"] = (
    model_df["Delivered to Client Date"]
    - model_df["Scheduled Delivery Date"]
).dt.days

model_df["Delayed"] = (
    model_df["Delay_Days"] > 0
).astype(int)

# Use only information available before delivery
features = [
    "Country",
    "Shipment Mode",
    "Vendor",
    "Product Group",
    "Line Item Quantity",
    "Pack Price",
    "Unit Price"
]

features = [c for c in features if c in model_df.columns]

X = model_df[features].copy()
y = model_df["Delayed"]

# Clean features
for col in X.select_dtypes(include=["object", "str"]).columns:
    X[col] = X[col].fillna("Unknown").astype(str)

for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].replace([np.inf, -np.inf], np.nan)
    X[col] = X[col].fillna(X[col].median())

# Convert text features to numeric columns
X = pd.get_dummies(X, dummy_na=False)

if y.nunique() < 2:
    print("Cannot train: the data contains only one delay class.")
elif y.value_counts().min() < 2:
    print("Cannot stratify: one class has fewer than 2 examples.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print("Model training completed!")
    print("Training rows:", len(X_train))
    print("Testing rows:", len(X_test))
    print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
    print("\nClassification report:")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

Model training completed!
Training rows: 8259
Testing rows: 2065
Accuracy: 0.841

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      1828
           1       0.36      0.48      0.41       237

    accuracy                           0.84      2065
   macro avg       0.64      0.68      0.66      2065
weighted avg       0.86      0.84      0.85      2065



In [18]:
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score
import numpy as np

# 1. Where are the errors?
print(confusion_matrix(y_test, predictions))

# 2. Find a better threshold than 0.5
probs = model.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, probs), 3))

prec, rec, thr = precision_recall_curve(y_test, probs)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = np.argmax(f1[:-1])
print("Best threshold:", round(thr[best], 3), "-> F1:", round(f1[best], 3))

# 3. Re-score with that threshold
from sklearn.metrics import classification_report
new_preds = (probs >= thr[best]).astype(int)
print(classification_report(y_test, new_preds, zero_division=0))

[[1622  206]
 [ 123  114]]
PR-AUC: 0.324
Best threshold: 0.31 -> F1: 0.434
              precision    recall  f1-score   support

           0       0.95      0.81      0.88      1828
           1       0.32      0.68      0.43       237

    accuracy                           0.80      2065
   macro avg       0.64      0.75      0.65      2065
weighted avg       0.88      0.80      0.83      2065



In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

m2 = RandomForestClassifier(
    n_estimators=400, min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
m2.fit(X_train, y_train)
p2 = m2.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, p2), 3))

# top features
import pandas as pd
print(pd.Series(m2.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False).head(10))

PR-AUC: 0.356
Line Item Quantity           0.204245
Pack Price                   0.173677
Unit Price                   0.124876
Vendor_SCMS from RDC         0.091374
Country_Vietnam              0.035371
Country_Nigeria              0.025348
Country_South Africa         0.024167
Shipment Mode_Truck          0.023488
Shipment Mode_Unknown        0.022585
Vendor_S. BUYS WHOLESALER    0.020331
dtype: float64


In [20]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

def add_feats(X):
    X = X.copy()
    X["total_value"] = X["Line Item Quantity"] * X["Unit Price"]
    X["price_ratio"] = X["Pack Price"] / (X["Unit Price"] + 1e-9)
    X["log_qty"] = np.log1p(X["Line Item Quantity"])
    X["log_value"] = np.log1p(X["total_value"])
    return X

Xtr, Xte = add_feats(X_train), add_feats(X_test)

m3 = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
m3.fit(Xtr, y_train)
p3 = m3.predict_proba(Xte)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, p3), 3))

PR-AUC: 0.341


In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

m4 = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=20, class_weight="balanced",
    early_stopping=True, random_state=42
)
m4.fit(X_train, y_train)
p4 = m4.predict_proba(X_test)[:, 1]
print("PR-AUC:", round(average_precision_score(y_test, p4), 3))

PR-AUC: 0.355


In [23]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=42)

for name, mdl in [("RandomForest m2", m2), ("HistGB m4", m4)]:
    s = cross_val_score(mdl, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    print(name, round(s.mean(), 3), "+/-", round(s.std(), 3))

RandomForest m2 0.337 +/- 0.028
HistGB m4 0.334 +/- 0.027


In [24]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix
import numpy as np

# 1. Out-of-fold probabilities on TRAIN only
oof = cross_val_predict(m2, X_train, y_train, cv=cv,
                        method="predict_proba", n_jobs=-1)[:, 1]

# 2. Threshold: best precision while catching >= 70% of class 1
prec, rec, thr = precision_recall_curve(y_train, oof)
ok = np.where(rec[:-1] >= 0.70)[0]
t = thr[ok[-1]]
print("Threshold:", round(t, 3))

# 3. Apply once to the test set
preds = (m2.predict_proba(X_test)[:, 1] >= t).astype(int)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, zero_division=0))

Threshold: 0.497
[[1390  438]
 [  54  183]]
              precision    recall  f1-score   support

           0       0.96      0.76      0.85      1828
           1       0.29      0.77      0.43       237

    accuracy                           0.76      2065
   macro avg       0.63      0.77      0.64      2065
weighted avg       0.89      0.76      0.80      2065

